**Notebook for analyzing WPINN model performance using simulated data**

Import libraries

In [5]:
import pandas as pd
import numpy as np
from utils.processing_functions import get_out_of_dist_split_indexes, moving_average
from models.conv_model import ConvModel
from models.pdp_model import PDPModel

Import data

In [65]:
# Read in data .pkl file
df = pd.read_pickle('data/example_data.pkl')
# Model inputs
x_flow = df['x_flow'][0]   # Simulated blood flow waveform
x_beat  = df['x_beat'][0]   # Simulated Bio-Z waveform
x_time = df['x_time'][0]   # Time array for waveforms
# Blood pressure (BP) output
y_BP = df['y_BP'][0]       # Simulated BP waveform

Train models and compare root mean squared errors
- Conventional data driven neural network (conv)
- Windkessel physics informed neural network (wpinn)

In [ ]:
 # Define parameter for analysis
Ew_model = 2    # Define (2)E or (3)E Windkessel model
num_epochs = 64 # Number of epochs for model training
w = 10          # Window size for moving average filter
N_iter = 5      # Number of iterations to re-run results

### Loop through multiple experiments to analyze performance ###
# Define type of BP to predict and analyze
for BP_type in ['SBP', 'DBP']:
    # Multiplier for percentile & batch size (mult:1, percentile=2.5, batch size=8; mult:2, percentile=5.0, batch size=16)
    for mult in [1, 2]:
        # Define arrays to append RMSE results to
        rmse_conv_arr, rmse_wpinn_arr = [], []
        # Iterate through define number of iterations
        for _ in range(N_iter):
            # Split indexes according to SBP, DBP, or MAP and the respective percentiles
            train_ind, val_ind, test_ind = get_out_of_dist_split_indexes(y_BP, BP_type=BP_type, percentile=2.5*mult)

            # Train conventional (CONV) model
            _, y_test, y_conv, _, _ = ConvModel(x_flow, x_time, x_beat, y_BP, train_ind, val_ind, test_ind).model_train(batch=8*mult, epochs=num_epochs)

            # Train WPINN model
            _, _, y_wpinn, _, _, _ = PDPModel(x_flow, x_time, x_beat, y_BP, train_ind, val_ind, test_ind).model_train(Ew_model=Ew_model, physics_weight=1,
                                                                                                                      batch=8*mult, epochs=num_epochs)

            # Evaluate for BP type
            if BP_type == 'DBP':  # DBP defined as starting BP value of beat
                dbp_conv = moving_average(y_conv[:, 0], w=w)
                dbp_wpinn = moving_average(y_wpinn[:, 0], w=w)
                dbp_test = moving_average(y_test[:, 0], w=w)
                rmse_conv = np.sqrt(np.mean(np.square(dbp_test - dbp_conv)))
                rmse_wpinn = np.sqrt(np.mean(np.square(dbp_test - dbp_wpinn)))

            elif BP_type == 'SBP': # SBP defined as peak value within BP waveform beat
                sbp_conv = moving_average(y_conv.max(axis=-1), w=w)
                sbp_wpinn = moving_average(y_wpinn.max(axis=-1), w=w)
                sbp_test = moving_average(y_test.max(axis=-1), w=w)
                rmse_conv = np.sqrt(np.mean(np.square(sbp_test - sbp_conv)))
                rmse_wpinn = np.sqrt(np.mean(np.square(sbp_test - sbp_wpinn)))

            # Append results
            rmse_conv_arr.append(rmse_conv)
            rmse_wpinn_arr.append(rmse_wpinn)

        # Convert to array
        rmse_conv_arr, rmse_wpinn_arr = np.array(rmse_conv_arr), np.array(rmse_wpinn_arr)

        # Print Results
        print(f'\nResults for {BP_type}, percentiles {2.5*mult}%')
        print(f'Conv model RMSE: {round(rmse_conv_arr.mean(), 2)} ± {round(rmse_conv_arr.std(), 2)} mmHg')
        print(f'WPINN model RMSE: {round(rmse_wpinn_arr.mean(), 2)} ± {round(rmse_wpinn_arr.std(), 2)} mmHg')


Results for SBP, percentiles 2.5%
Conv model RMSE: 16.35 ± 2.51 mmHg
WPINN model RMSE: 7.33 ± 2.86 mmHg
